Importing of libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Load dataset
data = pd.read_csv('data.csv')

In [ ]:
# Display first few rows of the dataset
print("First 5 rows of the dataset:")
print(data.head())

# Displaying the columns of the dataset
print("All columns in the dataset:")
print(data.columns)

In [ ]:
# Check for missing values
missing_values = data.isnull().sum()
print("Missing values in each column:")
print(missing_values)

In [ ]:
# Drop unwanted columns
data.drop(['column_name1', 'column_name2'], axis=1, inplace=True)


In [ ]:
# Imputation for missing values
num_cols = data.select_dtypes(include=['float64', 'int64']).columns
cat_cols = data.select_dtypes(include=['object']).columns

num_imputer = SimpleImputer(strategy='mean')
cat_imputer = SimpleImputer(strategy='most_frequent')

In [ ]:
# Apply imputers
data[num_cols] = num_imputer.fit_transform(data[num_cols])
data[cat_cols] = cat_imputer.fit_transform(data[cat_cols])

In [ ]:
# Handle outliers
def handle_outliers(data, threshold=3):
    mean = np.mean(data)
    std_dev = np.std(data)
    outliers = (data - mean).abs() > threshold * std_dev
    cleaned_data = data.mask(outliers)
    return cleaned_data


# Select the column containing the numerical data you want to handle outliers for
# Example: replace 'your_column_name' with your actual column name
column_name = 'your_column_name'

# Handle outliers for the selected column
data[column_name] = handle_outliers(data[column_name])

In [ ]:
# Visualize the data before and after handling outliers
plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
plt.hist(data[column_name].dropna(), bins=20, color='blue', alpha=0.5)
plt.title('Data with Outliers')

plt.subplot(2, 1, 2)
plt.hist(data[column_name].dropna(), bins=20, color='green', alpha=0.5)
plt.title('Data after Outlier Handling')

plt.tight_layout()
plt.show()

In [ ]:
# Feature scaling
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(data[num_cols])
scaled_df = pd.DataFrame(scaled_features, columns=num_cols)

# Standardizing a single column
scaler = StandardScaler()
data['standardized_column'] = scaler.fit_transform(data[['column_to_standardize']])

In [ ]:
# Encoding categorical variables
label_encoder = LabelEncoder()
for col in cat_cols:
    data[col] = label_encoder.fit_transform(data[col])

# OneHotEncoder for categorical variables
one_hot_encoder = OneHotEncoder(sparse=False)
encoded_features = one_hot_encoder.fit_transform(data[cat_cols])
encoded_df = pd.DataFrame(encoded_features, columns=one_hot_encoder.get_feature_names_out(cat_cols))
data = pd.concat([data.drop(columns=cat_cols), encoded_df], axis=1)

In [ ]:
# Date-time features
data['date'] = pd.to_datetime(data['date_column'])  # Convert to datetime format
data['year'] = data['date'].dt.year
data['month'] = data['date'].dt.month
data['day'] = data['date'].dt.day

In [ ]:
# Additional feature engineering
data['interaction_feature'] = data['feature1'] * data['feature2']

In [ ]:
# Split the dataset into features and target variable
X = data.drop('target_column', axis=1)
y = data['target_column']

In [ ]:
# Principal Component Analysis (PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
principal_components = pca.fit_transform(X_scaled)
principal_df = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])

explained_variance_ratio = pca.explained_variance_ratio_
print("Explained variance ratio:", explained_variance_ratio)

plt.figure(figsize=(8, 6))
plt.bar(range(len(explained_variance_ratio)), explained_variance_ratio, alpha=0.5, align='center')
plt.xlabel('Principal Components')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance Ratio of Principal Components')
plt.show()

plt.scatter(principal_df['PC1'], principal_df['PC2'], c=y, cmap='viridis')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA Visualization')
plt.colorbar()
plt.show()

In [ ]:
# t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)

plt.figure(figsize=(10, 8))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='viridis')
plt.colorbar()
plt.title('t-SNE Visualization')
plt.xlabel('First t-SNE Component')
plt.ylabel('Second t-SNE Component')
plt.show()